In [97]:
import os
import openai
os.environ["OPENAI_API_KEY"] = "" #모두의 연구소에서 발급
openai.api_key = os.getenv("OPENAI_API_KEY")

In [488]:
stock_price_function = {
    "name": "arrange_stock_price",
    "description": "function that arrange a list of stock price and analyze stock price",
    "parameters": {
        "type": "object",
        "properties": {
            "stockcodes": {
                "type": "array",
                "description": "주식종목코드들",
                "items": {
                    "type": "object",
                    "description": "주식종목코드",
                    "properties": {
                        "stockcode": {
                            "type": "string",
                        },
                        "stockinfos": {
                            "type": "array",
                            "description": "주식가격정보",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "Date": {
                                        "type": "string",
                                        "description": "날짜"
                                    },
                                    "Open": {
                                        "type": "string",
                                        "description": "시작가"
                                    },
                                    "High": {
                                        "type": "string",
                                        "description": "최고가"
                                    },
                                    "Low": {
                                        "type": "string",
                                        "description": "최저가"
                                    },
                                    "Close": {
                                        "type": "string",
                                        "description": "종가"
                                    },
                                    "Volume": {
                                        "type": "string",
                                        "description": "거래량"
                                    },
                                },
                                "required": ["Date", "Open", "High", "Low", "Close", "Volume"],
                            },
                        },
                    },
                    "required": ["stockcode", "stockinfos"],
                },
            }
        },
        "required": ["stockcodes"],
    },
}

get_stock_price_function = {
    "name": "get_stock_price",
    "description": "function that return a price of stock of specific data",
    "parameters": {
        "type": "object",
        "properties": {
            "stockcode": {
                "type": "string",
                "description": "주식종목코드",
            },
            "Date": {
                "type": "string",
                "description": "날짜",
            },
            "Open": {
                "type": "string",
                "description": "시작가"
            },
            "High": {
                "type": "string",
                "description": "최고가"
            },
            "Low": {
                "type": "string",
                "description": "최저가"
            },
            "Close": {
                "type": "string",
                "description": "종가"
            },
            "Volume": {
                "type": "string",
                "description": "거래량"
            },
        }
    }
}

In [489]:
import json
from langchain.chat_models.openai import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.memory import ConversationSummaryBufferMemory
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder



chat_llm = ChatOpenAI(
    temperature=0.1,
).bind(
    function_call="auto",  # 수정된 부분
    functions=[stock_price_function, get_stock_price_function]
)


memory_llm = ChatOpenAI(
    temperature=0.1,
)

memory = ConversationSummaryBufferMemory(
    llm=memory_llm,
    max_token_limit=50,
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a chatbot that helps people with their daily life."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

def load_memory(input):
    history = memory.load_memory_variables({})['history']
    return history

chain = RunnablePassthrough.assign(history=load_memory) | prompt | chat_llm

def invoke_chain(question):
    result = chain.invoke({"question": question})
    memory.save_context(
        {"input": question},
        {"output": result.content},
    )
    return result

In [300]:
response = invoke_chain("오늘 한화시스템 주식정보는 어때?")
print(response)

content='' additional_kwargs={'function_call': {'arguments': '{"stockcode":"006280"}', 'name': 'get_stock_price'}} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 352, 'total_tokens': 369}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'function_call', 'logprobs': None} id='run-c3156c8b-515f-4725-8309-34a5b03efb43-0'


In [ ]:
!pip install yfinance

In [126]:
import yfinance as yf

In [508]:
stock = yf.Ticker("005930.KS")  # 삼성전자 종목 코드(KS는 한국 주식 시장을 의미)
data = stock.history(period="1mo")  # 지난 1개월 간의 데이터 가져오기

In [520]:
print(data)

                              Open     High      Low    Close    Volume  \
Date                                                                      
2024-07-16 00:00:00+09:00  86900.0  88000.0  86700.0  87700.0  16166688   
2024-07-17 00:00:00+09:00  87100.0  88000.0  86400.0  86700.0  18186490   
2024-07-18 00:00:00+09:00  83800.0  86900.0  83800.0  86900.0  24721790   
2024-07-19 00:00:00+09:00  85600.0  86100.0  84100.0  84400.0  18569122   
2024-07-22 00:00:00+09:00  84400.0  84900.0  82600.0  83000.0  18987560   
2024-07-23 00:00:00+09:00  84200.0  84700.0  83400.0  83900.0  15766389   
2024-07-24 00:00:00+09:00  82900.0  83300.0  81900.0  82000.0  16939083   
2024-07-25 00:00:00+09:00  80400.0  81000.0  80100.0  80400.0  20323811   
2024-07-26 00:00:00+09:00  80700.0  81300.0  80400.0  80900.0  14508334   
2024-07-29 00:00:00+09:00  81600.0  82000.0  81100.0  81200.0  12797136   
2024-07-30 00:00:00+09:00  80400.0  81000.0  80000.0  81000.0  13169636   
2024-07-31 00:00:00+09:00

In [510]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

In [511]:
def get_stock_price(stockcode, Date, Close, datalist):
    target_stockcode = stockcode
    target_date = Date

    # stockcode와 Date에 해당하는 Close 값을 찾기
    stock = next((item for item in datalist['stockcodes'] if item['stockcode'] == target_stockcode), None)
    if stock:
        info = next((info for info in stock['stockinfos'] if info['Date'] == target_date), None)
        close_value = info['Close'] if info else None
    else:
        close_value = None
     
    #print("close_value = " + close_value)   
     
    stock_info = {
        "stockcode": stockcode,
        "Date": Date,
        "Close": close_value
    }
    return json.dumps(stock_info)


In [289]:
# target_stockcode = '005930.KS'
# target_date = '2024-07-24'
# data = r

# # stockcode와 Date에 해당하는 Close 값을 찾기
# stock = next((item for item in data['stockcodes'] if item['stockcode'] == target_stockcode), None)
# if stock:
#     info = next((info for info in stock['stockinfos'] if info['Date'] == target_date), None)
#     close_value = info['Close'] if info else None
# else:
#     close_value = None

In [512]:
llm = ChatOpenAI(
    temperature= 0.1,
).bind(
    #function_call={"name": "get_weather"}, # 모델이 강제로 함수를 사용하도록
    function_call="auto", # 모델이 필요에 따라 함소호출여부를 선택하여 사용하도록
    #function_call={
    #    "name": "arrange_stock_price",
    #},
    functions=[
        stock_price_function, get_stock_price_function
    ]
)

In [513]:
# 주식 데이터를 문자열로 변환
data_str = data.to_string()

# PromptTemplate 생성
prompt = PromptTemplate.from_template("Make a list of prices about {stockcode}.\n\nHere is the stock data:\n{data}")



In [514]:
data_str

'                              Open     High      Low    Close    Volume  Dividends  Stock Splits\nDate                                                                                            \n2024-07-16 00:00:00+09:00  86900.0  88000.0  86700.0  87700.0  16166688        0.0           0.0\n2024-07-17 00:00:00+09:00  87100.0  88000.0  86400.0  86700.0  18186490        0.0           0.0\n2024-07-18 00:00:00+09:00  83800.0  86900.0  83800.0  86900.0  24721790        0.0           0.0\n2024-07-19 00:00:00+09:00  85600.0  86100.0  84100.0  84400.0  18569122        0.0           0.0\n2024-07-22 00:00:00+09:00  84400.0  84900.0  82600.0  83000.0  18987560        0.0           0.0\n2024-07-23 00:00:00+09:00  84200.0  84700.0  83400.0  83900.0  15766389        0.0           0.0\n2024-07-24 00:00:00+09:00  82900.0  83300.0  81900.0  82000.0  16939083        0.0           0.0\n2024-07-25 00:00:00+09:00  80400.0  81000.0  80100.0  80400.0  20323811        0.0           0.0\n2024-07-26 00:00:00

In [515]:
chain = prompt | llm

In [516]:
response_stcklist = chain.invoke({
    "stockcode":"005930.KS", "data": data_str
})

In [517]:
response_stcklist = response_stcklist.additional_kwargs["function_call"]["arguments"]

In [519]:
response_stcklist

'{"stockcodes":[{"stockcode":"005930.KS","stockinfos":[{"Date":"2024-07-16","Open":"86900.0","High":"88000.0","Low":"86700.0","Close":"87700.0","Volume":"16166688"},{"Date":"2024-07-17","Open":"87100.0","High":"88000.0","Low":"86400.0","Close":"86700.0","Volume":"18186490"},{"Date":"2024-07-18","Open":"83800.0","High":"86900.0","Low":"83800.0","Close":"86900.0","Volume":"24721790"},{"Date":"2024-07-19","Open":"85600.0","High":"86100.0","Low":"84100.0","Close":"84400.0","Volume":"18569122"},{"Date":"2024-07-22","Open":"84400.0","High":"84900.0","Low":"82600.0","Close":"83000.0","Volume":"18987560"},{"Date":"2024-07-23","Open":"84200.0","High":"84700.0","Low":"83400.0","Close":"83900.0","Volume":"15766389"},{"Date":"2024-07-24","Open":"82900.0","High":"83300.0","Low":"81900.0","Close":"82000.0","Volume":"16939083"},{"Date":"2024-07-25","Open":"80400.0","High":"81000.0","Low":"80100.0","Close":"80400.0","Volume":"20323811"},{"Date":"2024-07-26","Open":"80700.0","High":"81300.0","Low":"804

In [522]:
import json

data = json.loads(response_stcklist)

In [525]:
data

{'stockcodes': [{'stockcode': '005930.KS',
   'stockinfos': [{'Date': '2024-07-16',
     'Open': '86900.0',
     'High': '88000.0',
     'Low': '86700.0',
     'Close': '87700.0',
     'Volume': '16166688'},
    {'Date': '2024-07-17',
     'Open': '87100.0',
     'High': '88000.0',
     'Low': '86400.0',
     'Close': '86700.0',
     'Volume': '18186490'},
    {'Date': '2024-07-18',
     'Open': '83800.0',
     'High': '86900.0',
     'Low': '83800.0',
     'Close': '86900.0',
     'Volume': '24721790'},
    {'Date': '2024-07-19',
     'Open': '85600.0',
     'High': '86100.0',
     'Low': '84100.0',
     'Close': '84400.0',
     'Volume': '18569122'},
    {'Date': '2024-07-22',
     'Open': '84400.0',
     'High': '84900.0',
     'Low': '82600.0',
     'Close': '83000.0',
     'Volume': '18987560'},
    {'Date': '2024-07-23',
     'Open': '84200.0',
     'High': '84700.0',
     'Low': '83400.0',
     'Close': '83900.0',
     'Volume': '15766389'},
    {'Date': '2024-07-24',
     'Open

In [526]:


chat_llm = ChatOpenAI(
    temperature=0.1,
).bind(
    function_call="auto",  # 수정된 부분
    functions=[stock_price_function, get_stock_price_function]
)


memory_llm = ChatOpenAI(
    temperature=0.1,
)

memory = ConversationSummaryBufferMemory(
    llm=memory_llm,
    max_token_limit=50,
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a chatbot that helps people with their daily life."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

def load_memory(input):
    history = memory.load_memory_variables({})['history']
    return history

chain = RunnablePassthrough.assign(history=load_memory) | prompt | chat_llm

def invoke_chain(question):
    result = chain.invoke({"question": question})
    memory.save_context(
        {"input": question},
        {"output": result.content},
    )
    return result

In [527]:
response = invoke_chain("005930.KS의 2024년 7월 24일 주식 종가정보를 알려주세요\n")
print(response)

content='' additional_kwargs={'function_call': {'arguments': '{"stockcode":"005930.KS","Date":"2024-07-24","Close":""}', 'name': 'get_stock_price'}} response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 294, 'total_tokens': 325}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'function_call', 'logprobs': None} id='run-e9b0645d-2c55-421c-acb9-5b2698ed387e-0'


In [528]:
def function_calling(input):
    available_functions = {
            "get_stock_price": get_stock_price,
        }

    function_call = input.additional_kwargs["function_call"]
    function_name = function_call["name"]
    function_to_call = available_functions.get(function_name)

    if function_to_call:
        # Extract arguments from the function call
        function_args = json.loads(function_call["arguments"])
        function_input = function_to_call(
            stockcode=function_args.get("stockcode"),
            Date=function_args.get("Date"),
            Close=function_args.get("Close"),
            datalist=data
        )

        result = invoke_chain(function_input)

        return result.content

    return "Not exist function call"



In [529]:
response.additional_kwargs.get("function_call")

{'arguments': '{"stockcode":"005930.KS","Date":"2024-07-24","Close":""}',
 'name': 'get_stock_price'}

In [532]:

if response.additional_kwargs.get("function_call"):
    result = function_calling(response)


In [533]:
result

'The closing stock price of 005930.KS on July 24, 2024, is 82000.0.'